# Gym Management System — OOP Simulation Project

**Course:** Event-Based Simulation, BGU — Industrial Engineering and Management
**Lecturer:** Nimrod Talmon | **TAs:** Amit Oren, Eden Cohen
**Team:** Ido, Yonatan, Eitan

---

## Project Overview

This notebook implements an OOP-based gym management system with 7 classes:

| # | Class | Role | Type |
|---|-------|------|------|
| 1 | `Equipment` | Gym equipment with malfunction/repair tracking | Extra (substantial) |
| 2 | `Subscription` | Billing and payment logic | Extra (lightweight) |
| 3 | `FitnessClass` | Group fitness classes with enrollment and waitlists | Required |
| 4 | `Trainer` | Trainers with scheduling and statistics | Required |
| 5 | `Member` | Gym members with enrollments and payments | Required |
| 6 | `PersonalTrainingSession` | 1-on-1 training sessions | Extra (substantial) |
| 7 | `Gym` | Top-level container with reports and search | Required |

Classes are defined in dependency order — each class only uses classes defined above it.

**Design decisions** (pricing, scales, schedule format, etc.) are explained in markdown cells throughout the notebook.

## 1. Equipment Class

We start with `Equipment` because other classes (like `FitnessClass`) depend on it, but it doesn't depend on anything else.

**Why this extra class?** Real gyms manage equipment, and classes depend on it. This adds a simulation-ready element: equipment can break down, which triggers warnings for affected classes (a cascade effect). Right now the breakdown is deterministic, but the architecture is ready for probabilistic breakdowns if we extend this later.

**Design decisions:**
- Auto-generated IDs with prefix `E` (e.g., E001, E002) using a class-level counter
- `condition` is one of: `"good"`, `"fair"`, `"needs_repair"`
- `report_malfunction()` prints warnings for all affected classes — but does NOT auto-cancel them (the admin decides)

**Known simplification:** We model equipment at the *type level*, not the individual unit level. All 25 yoga mats share one `condition` field — if one breaks, they're all marked as `"needs_repair"`. A more granular design would track each unit separately and reduce `quantity_available` when one breaks. We kept it simple because the project focuses on class interactions and cascading events, not inventory management. This would be easy to extend if needed.

In [1]:
class Equipment:
    """Represents a piece of gym equipment (e.g., treadmill, bike, mat)."""

    _id_counter = 0
    VALID_CONDITIONS = ["good", "fair", "needs_repair"]

    def __init__(self, name, equipment_type, quantity_available, condition="good"):
        # Validate inputs
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Equipment name must be a non-empty string.")
        if not isinstance(equipment_type, str) or not equipment_type.strip():
            raise ValueError("Equipment type must be a non-empty string.")
        if not isinstance(quantity_available, int) or quantity_available < 0:
            raise ValueError("Quantity must be a non-negative integer.")
        if condition not in Equipment.VALID_CONDITIONS:
            raise ValueError(f"Condition must be one of {Equipment.VALID_CONDITIONS}.")

        Equipment._id_counter += 1
        self._equipment_id = f"E{Equipment._id_counter:03d}"
        self._name = name
        self._equipment_type = equipment_type
        self._quantity_available = quantity_available
        self._condition = condition
        self._assigned_to_classes = []  # list of FitnessClass objects

    # --- Properties ---

    @property
    def equipment_id(self):
        return self._equipment_id

    @property
    def name(self):
        return self._name

    @property
    def equipment_type(self):
        return self._equipment_type

    @property
    def quantity_available(self):
        return self._quantity_available

    @property
    def condition(self):
        return self._condition

    @property
    def assigned_to_classes(self):
        return self._assigned_to_classes

    # --- Methods ---

    def is_available(self, quantity_needed=1):
        """Check if enough units are available and equipment is not broken."""
        if not isinstance(quantity_needed, int) or quantity_needed < 1:
            raise ValueError("Quantity needed must be a positive integer.")
        return self._condition != "needs_repair" and self._quantity_available >= quantity_needed

    def reserve(self, fitness_class):
        """Assign this equipment to a fitness class."""
        if fitness_class in self._assigned_to_classes:
            print(f"  Warning: {self._name} is already assigned to {fitness_class.name}.")
            return False
        if not self.is_available():
            print(f"  Failed: {self._name} is not available (condition: {self._condition}, qty: {self._quantity_available}).")
            return False
        self._assigned_to_classes.append(fitness_class)
        print(f"  OK: {self._name} reserved for class '{fitness_class.name}'.")
        return True

    def release(self, fitness_class):
        """Free equipment from a fitness class."""
        if fitness_class not in self._assigned_to_classes:
            print(f"  Failed: {self._name} is not assigned to '{fitness_class.name}'.")
            return False
        self._assigned_to_classes.remove(fitness_class)
        print(f"  OK: {self._name} released from class '{fitness_class.name}'.")
        return True

    def report_malfunction(self):
        """Mark equipment as needing repair and warn about affected classes."""
        self._condition = "needs_repair"
        print(f"\nMALFUNCTION REPORTED: {self._name} ({self._equipment_id})")
        if self._assigned_to_classes:
            print(f"  WARNING - The following classes are affected:")
            for fc in self._assigned_to_classes:
                print(f"    - {fc.name} (ID: {fc.class_id})")
        else:
            print("  No classes currently using this equipment.")

    def repair(self):
        """Restore equipment to working condition."""
        self._condition = "good"
        print(f"  OK: {self._name} has been repaired and is back in service.")

    def get_usage_report(self):
        """Print a report on how this equipment is used across classes."""
        print(f"\n--- Equipment Usage Report: {self._name} ({self._equipment_id}) ---")
        print(f"  Type: {self._equipment_type}")
        print(f"  Condition: {self._condition}")
        print(f"  Quantity available: {self._quantity_available}")
        print(f"  Assigned to {len(self._assigned_to_classes)} class(es):")
        if self._assigned_to_classes:
            for fc in self._assigned_to_classes:
                print(f"    - {fc.name}")
        else:
            print("    (none)")

    def __str__(self):
        return (f"Equipment({self._equipment_id}: {self._name}, "
                f"type={self._equipment_type}, qty={self._quantity_available}, "
                f"condition={self._condition})")

    def __eq__(self, other):
        if not isinstance(other, Equipment):
            return False
        return self._equipment_id == other._equipment_id

### Equipment — Usage Examples

In [2]:
# --- Equipment usage examples ---

# Create equipment
bikes = Equipment("Spinning Bikes", "bike", 10)
mats = Equipment("Yoga Mats", "mat", 20)
print(bikes)
print(mats)

# Check availability
print(f"\nBikes available (need 5): {bikes.is_available(5)}")
print(f"Bikes available (need 15): {bikes.is_available(15)}")

# Malfunction and repair
bikes.report_malfunction()
print(f"Bikes available after malfunction: {bikes.is_available()}")
bikes.repair()
print(f"Bikes available after repair: {bikes.is_available()}")

# Usage report (no classes assigned yet)
bikes.get_usage_report()

# Input validation
try:
    bad = Equipment("", "type", 5)
except ValueError as e:
    print(f"\nValidation error: {e}")

# reserve() and release() will be fully demoed with FitnessClass objects later,
# but here's the basic flow with a minimal example:
class _TempClass:
    """Temporary stand-in to demo reserve/release before FitnessClass is defined."""
    def __init__(self, name):
        self.name = name
        self.class_id = "TEMP"

temp_class = _TempClass("Temp Yoga")
mats.reserve(temp_class)
mats.release(temp_class)
del _TempClass, temp_class

# Reset counter for next sections
Equipment._id_counter = 0

Equipment(E001: Spinning Bikes, type=bike, qty=10, condition=good)
Equipment(E002: Yoga Mats, type=mat, qty=20, condition=good)

Bikes available (need 5): True
Bikes available (need 15): False

MALFUNCTION REPORTED: Spinning Bikes (E001)
  No classes currently using this equipment.
Bikes available after malfunction: False
  OK: Spinning Bikes has been repaired and is back in service.
Bikes available after repair: True

--- Equipment Usage Report: Spinning Bikes (E001) ---
  Type: bike
  Condition: good
  Quantity available: 10
  Assigned to 0 class(es):
    (none)

Validation error: Equipment name must be a non-empty string.
  OK: Yoga Mats reserved for class 'Temp Yoga'.
  OK: Yoga Mats released from class 'Temp Yoga'.


## 2. Subscription Class

**Why this extra class?** The assignment requires a monthly payment report with pricing based on plan type, class counts, and no-show penalties. Without `Subscription`, all that billing logic would be crammed into `Member.generate_payment_report()`. We separated it to keep each class focused on one responsibility.

This is intentionally a lightweight class — about 4 fields and 4 methods. It's a helper, not a core simulation object.

**Pricing model (our choice — assignment says weights are up to us):**
- Monthly: 250 NIS/month, includes 8 classes
- Yearly: 200 NIS/month, includes 12 classes
- Premium: 350 NIS/month, unlimited classes
- Extra class beyond plan: 30 NIS each
- No-show penalty: 25 NIS per missed class

In [3]:
class Subscription:
    """Handles billing and payment logic for a gym member."""

    # Pricing configuration
    PLAN_DETAILS = {
        "monthly": {"price": 250, "classes_included": 8},
        "yearly":  {"price": 200, "classes_included": 12},
        "premium": {"price": 350, "classes_included": float('inf')}  # unlimited
    }
    EXTRA_CLASS_FEE = 30   # NIS per class beyond the included amount
    NO_SHOW_PENALTY = 25   # NIS per no-show

    def __init__(self, member, plan_type):
        if plan_type not in Subscription.PLAN_DETAILS:
            raise ValueError(f"Plan type must be one of {list(Subscription.PLAN_DETAILS.keys())}.")

        self._member = member
        self._plan_type = plan_type
        details = Subscription.PLAN_DETAILS[plan_type]
        self._monthly_price = details["price"]
        self._classes_included = details["classes_included"]
        self._is_active = True

    @property
    def member(self):
        return self._member

    @property
    def plan_type(self):
        return self._plan_type

    @property
    def monthly_price(self):
        return self._monthly_price

    @property
    def classes_included(self):
        return self._classes_included

    @property
    def is_active(self):
        return self._is_active

    def calculate_monthly_bill(self, classes_attended, no_shows):
        """Compute the monthly bill: base price + overage charges + no-show penalties."""
        if not isinstance(classes_attended, int) or classes_attended < 0:
            raise ValueError("Classes attended must be a non-negative integer.")
        if not isinstance(no_shows, int) or no_shows < 0:
            raise ValueError("No-shows must be a non-negative integer.")

        bill = self._monthly_price

        # Overage: classes beyond what the plan includes (premium = unlimited)
        if self._classes_included != float('inf'):
            extra_classes = max(0, classes_attended - self._classes_included)
            bill += extra_classes * Subscription.EXTRA_CLASS_FEE

        # No-show penalties
        bill += no_shows * Subscription.NO_SHOW_PENALTY
        return bill

    def cancel(self):
        """Deactivate the subscription."""
        if not self._is_active:
            print(f"  Subscription for {self._member.name} is already cancelled.")
            return
        self._is_active = False
        print(f"  OK: Subscription cancelled for {self._member.name}.")

    def renew(self):
        """Reactivate the subscription."""
        if self._is_active:
            print(f"  Subscription for {self._member.name} is already active.")
            return
        self._is_active = True
        print(f"  OK: Subscription renewed for {self._member.name}.")

    def __str__(self):
        status = "Active" if self._is_active else "Cancelled"
        included = "unlimited" if self._classes_included == float('inf') else self._classes_included
        return (f"Subscription({self._member.name}: {self._plan_type}, "
                f"{self._monthly_price} NIS/month, "
                f"{included} classes included, {status})")

### Subscription — Usage Examples

We can't fully demo Subscription yet because it needs a Member object. We'll create a minimal mock to show the billing logic, and see the full interaction later.

In [4]:
# --- Subscription usage examples ---

# Minimal object with .name attribute for testing
class _TempMember:
    def __init__(self, name):
        self.name = name

temp = _TempMember("Test User")

# Create subscriptions for each plan type
sub_monthly = Subscription(temp, "monthly")
sub_yearly = Subscription(temp, "yearly")
sub_premium = Subscription(temp, "premium")
print(sub_monthly)
print(sub_yearly)
print(sub_premium)

# Billing examples
print(f"\nMonthly plan, 8 classes, 0 no-shows: {sub_monthly.calculate_monthly_bill(8, 0)} NIS")
print(f"Monthly plan, 10 classes, 0 no-shows: {sub_monthly.calculate_monthly_bill(10, 0)} NIS")
print(f"Monthly plan, 10 classes, 2 no-shows: {sub_monthly.calculate_monthly_bill(10, 2)} NIS")
print(f"Premium plan, 20 classes, 1 no-show: {sub_premium.calculate_monthly_bill(20, 1)} NIS")

# Cancel and renew
sub_monthly.cancel()
print(sub_monthly)
sub_monthly.renew()
print(sub_monthly)

# Validation
try:
    bad_sub = Subscription(temp, "gold")
except ValueError as e:
    print(f"\nValidation error: {e}")

del _TempMember, temp

Subscription(Test User: monthly, 250 NIS/month, 8 classes included, Active)
Subscription(Test User: yearly, 200 NIS/month, 12 classes included, Active)
Subscription(Test User: premium, 350 NIS/month, unlimited classes included, Active)

Monthly plan, 8 classes, 0 no-shows: 250 NIS
Monthly plan, 10 classes, 0 no-shows: 310 NIS
Monthly plan, 10 classes, 2 no-shows: 360 NIS
Premium plan, 20 classes, 1 no-show: 375 NIS
  OK: Subscription cancelled for Test User.
Subscription(Test User: monthly, 250 NIS/month, 8 classes included, Cancelled)
  OK: Subscription renewed for Test User.
Subscription(Test User: monthly, 250 NIS/month, 8 classes included, Active)

Validation error: Plan type must be one of ['monthly', 'yearly', 'premium'].


## 3. FitnessClass

**Design decisions:**
- **Schedule slots** are tuples like `("Sunday", "10:00")`. The Israeli work week runs Sunday-Thursday + Friday morning. Time slots are on the hour only.
- **Conflict detection:** Two classes conflict if they're on the same day and their time ranges overlap (based on start time + duration).
- **Fitness suitability:** A member can join if the class difficulty is **at most 1 level above** their fitness level. Both use a 1-5 integer scale.
- **Equipment mapping:** A dictionary maps class category to required equipment type (e.g., spinning -> bike, yoga -> mat). When a FitnessClass is created, it automatically looks up what equipment it needs.
- **Waitlist:** When a class is full, members go on the waiting list. When someone drops out, the first waitlisted member is automatically promoted (if still suitable).

In [5]:
class FitnessClass:
    """Represents a group fitness class offered by the gym."""

    _id_counter = 0
    VALID_DAYS = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

    # Maps class category to required equipment type
    EQUIPMENT_MAP = {
        "spinning": "bike",
        "yoga": "mat",
        "pilates": "mat",
        "crossfit": "weights",
        "boxing": "punching_bag",
        "strength": "weights",
    }

    def __init__(self, name, category, difficulty_level, capacity, duration=60):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Class name must be a non-empty string.")
        if not isinstance(category, str) or not category.strip():
            raise ValueError("Category must be a non-empty string.")
        if not isinstance(difficulty_level, int) or not 1 <= difficulty_level <= 5:
            raise ValueError("Difficulty level must be an integer between 1 and 5.")
        if not isinstance(capacity, int) or capacity < 1:
            raise ValueError("Capacity must be a positive integer.")
        if not isinstance(duration, int) or duration < 1:
            raise ValueError("Duration must be a positive integer (minutes).")

        FitnessClass._id_counter += 1
        self._class_id = f"C{FitnessClass._id_counter:03d}"
        self._name = name
        self._category = category.lower()
        self._difficulty_level = difficulty_level
        self._capacity = capacity
        self._duration = duration
        self._trainer = None
        self._schedule_slot = None  # tuple: (day, time)
        self._enrolled_members = []
        self._waiting_list = []
        self._required_equipment_type = FitnessClass.EQUIPMENT_MAP.get(self._category, None)

    # --- Properties ---

    @property
    def class_id(self):
        return self._class_id

    @property
    def name(self):
        return self._name

    @property
    def category(self):
        return self._category

    @property
    def difficulty_level(self):
        return self._difficulty_level

    @property
    def capacity(self):
        return self._capacity

    @property
    def duration(self):
        return self._duration

    @property
    def trainer(self):
        return self._trainer

    @property
    def schedule_slot(self):
        return self._schedule_slot

    @property
    def enrolled_members(self):
        return self._enrolled_members

    @property
    def waiting_list(self):
        return self._waiting_list

    @property
    def required_equipment_type(self):
        return self._required_equipment_type

    # --- Helper: time overlap detection ---

    def _get_end_time(self):
        """Calculate end time based on schedule slot and duration."""
        if self._schedule_slot is None:
            return None
        hour, minute = map(int, self._schedule_slot[1].split(":"))
        total_minutes = hour * 60 + minute + self._duration
        end_hour = total_minutes // 60
        end_min = total_minutes % 60
        return f"{end_hour:02d}:{end_min:02d}"

    def overlaps_with(self, other_slot, other_duration=60):
        """Check if this class's schedule overlaps with another time slot."""
        if self._schedule_slot is None:
            return False
        if self._schedule_slot[0] != other_slot[0]:  # different day
            return False

        # Convert to minutes for easy comparison
        my_start = int(self._schedule_slot[1].split(":")[0]) * 60 + int(self._schedule_slot[1].split(":")[1])
        my_end = my_start + self._duration
        other_start = int(other_slot[1].split(":")[0]) * 60 + int(other_slot[1].split(":")[1])
        other_end = other_start + other_duration

        return my_start < other_end and other_start < my_end

    # --- Methods ---

    def assign_trainer(self, trainer):
        """Link a trainer to this class."""
        self._trainer = trainer
        print(f"  OK: Trainer '{trainer.name}' assigned to class '{self._name}'.")

    def set_schedule(self, schedule_slot):
        """Set the day and time for this class. Expects a tuple like ('Sunday', '10:00')."""
        if not isinstance(schedule_slot, tuple) or len(schedule_slot) != 2:
            raise ValueError("Schedule slot must be a tuple of (day, time).")
        day, time = schedule_slot
        if day not in FitnessClass.VALID_DAYS:
            raise ValueError(f"Day must be one of {FitnessClass.VALID_DAYS}.")
        parts = time.split(":")
        if len(parts) != 2 or not parts[0].isdigit() or not parts[1].isdigit():
            raise ValueError("Time must be in HH:MM format.")
        hour, minute = int(parts[0]), int(parts[1])
        if not (6 <= hour <= 21) or minute != 0:
            raise ValueError("Time must be on the hour, between 06:00 and 21:00.")

        self._schedule_slot = schedule_slot
        print(f"  OK: Class '{self._name}' scheduled for {day} at {time}.")

    def has_capacity(self):
        """Returns True if there's room for more members."""
        return len(self._enrolled_members) < self._capacity

    def is_suitable(self, member):
        """Check if difficulty matches the member's fitness level.
        Rule: member can join if difficulty <= fitness_level + 1."""
        return self._difficulty_level <= member.fitness_level + 1

    def enroll(self, member):
        """Add a member to this class (checks capacity, suitability, schedule conflicts)."""
        if member in self._enrolled_members:
            print(f"  Warning: {member.name} is already enrolled in '{self._name}'.")
            return False

        if not self.is_suitable(member):
            print(f"  REJECTED: {member.name} (fitness level {member.fitness_level}) "
                  f"cannot join '{self._name}' (difficulty {self._difficulty_level}). "
                  f"Requires at least fitness level {self._difficulty_level - 1}.")
            return False

        if self._schedule_slot and not member.can_attend(self._schedule_slot, self._duration):
            print(f"  REJECTED: {member.name} has a schedule conflict at {self._schedule_slot}.")
            return False

        if not self.has_capacity():
            print(f"  FULL: Class '{self._name}' is full. Adding {member.name} to waiting list.")
            self.add_to_waiting_list(member)
            return False

        self._enrolled_members.append(member)
        member.current_enrollments.append(self)
        print(f"  OK: {member.name} enrolled in '{self._name}'.")
        return True

    def drop(self, member_id):
        """Remove a member by ID and try to promote from the waiting list."""
        for member in self._enrolled_members:
            if member.member_id == member_id:
                self._enrolled_members.remove(member)
                member.current_enrollments.remove(self)
                print(f"  OK: {member.name} dropped from '{self._name}'.")
                self.promote_from_waiting_list()
                return True
        print(f"  Failed: No member with ID '{member_id}' found in '{self._name}'.")
        return False

    def add_to_waiting_list(self, member):
        """Add a member to the waiting list."""
        if member in self._waiting_list:
            print(f"  Warning: {member.name} is already on the waiting list for '{self._name}'.")
            return False
        self._waiting_list.append(member)
        print(f"  OK: {member.name} added to waiting list for '{self._name}'.")
        return True

    def promote_from_waiting_list(self):
        """Move the first suitable waitlisted member into the class."""
        while self._waiting_list and self.has_capacity():
            candidate = self._waiting_list.pop(0)
            if self.is_suitable(candidate):
                self._enrolled_members.append(candidate)
                candidate.current_enrollments.append(self)
                print(f"  PROMOTED: {candidate.name} promoted from waiting list to '{self._name}'.")
                return True
            else:
                print(f"  Warning: {candidate.name} on waiting list is no longer suitable - skipped.")
        return False

    def __str__(self):
        trainer_name = self._trainer.name if self._trainer else "Unassigned"
        schedule = f"{self._schedule_slot[0]} {self._schedule_slot[1]}" if self._schedule_slot else "Unscheduled"
        return (f"FitnessClass({self._class_id}: '{self._name}', "
                f"category={self._category}, difficulty={self._difficulty_level}, "
                f"trainer={trainer_name}, schedule={schedule}, "
                f"enrolled={len(self._enrolled_members)}/{self._capacity})")

    def __eq__(self, other):
        if not isinstance(other, FitnessClass):
            return False
        return self._class_id == other._class_id

### FitnessClass — Usage Examples

Full enrollment/waitlist demos need Member and Trainer objects (defined below). Here we test basic creation, scheduling, and validation.

In [6]:
# --- FitnessClass usage examples ---

yoga = FitnessClass("Morning Yoga", "yoga", difficulty_level=2, capacity=15)
print(yoga)

yoga.set_schedule(("Sunday", "10:00"))
print(yoga)

# assign_trainer() - direct call (also called internally by Trainer.assign_class)
class _TempTrainer:
    def __init__(self, name):
        self.name = name
temp_trainer = _TempTrainer("Test Trainer")
yoga.assign_trainer(temp_trainer)
del _TempTrainer, temp_trainer

# Equipment mapping - yoga category automatically maps to "mat"
print(f"\nRequired equipment for yoga: {yoga.required_equipment_type}")

# has_capacity() - class was just created with 0/15 enrolled
print(f"\nhas_capacity (0/15 enrolled): {yoga.has_capacity()}")

# is_suitable() - needs a member-like object with fitness_level attribute
class _TempMember:
    def __init__(self, name, fitness_level):
        self.name = name
        self.fitness_level = fitness_level

beginner = _TempMember("Beginner", 1)
expert = _TempMember("Expert", 5)
# Yoga difficulty=2: member can join if 2 <= fitness + 1
print(f"is_suitable for beginner (fitness=1, difficulty=2): {yoga.is_suitable(beginner)}")  # 2 <= 1+1 = True
print(f"is_suitable for expert (fitness=5, difficulty=2): {yoga.is_suitable(expert)}")      # 2 <= 5+1 = True
del _TempMember, beginner, expert

# Overlap detection
print(f"\nOverlaps with Sunday 10:00 (60 min): {yoga.overlaps_with(('Sunday', '10:00'), 60)}")
print(f"Overlaps with Sunday 11:00 (60 min): {yoga.overlaps_with(('Sunday', '11:00'), 60)}")
print(f"Overlaps with Monday 10:00 (60 min): {yoga.overlaps_with(('Monday', '10:00'), 60)}")

# Validation
try:
    bad = FitnessClass("Bad", "yoga", difficulty_level=6, capacity=10)
except ValueError as e:
    print(f"\nValidation error: {e}")

try:
    yoga.set_schedule(("Saturday", "10:00"))
except ValueError as e:
    print(f"Validation error: {e}")

FitnessClass._id_counter = 0

FitnessClass(C001: 'Morning Yoga', category=yoga, difficulty=2, trainer=Unassigned, schedule=Unscheduled, enrolled=0/15)
  OK: Class 'Morning Yoga' scheduled for Sunday at 10:00.
FitnessClass(C001: 'Morning Yoga', category=yoga, difficulty=2, trainer=Unassigned, schedule=Sunday 10:00, enrolled=0/15)
  OK: Trainer 'Test Trainer' assigned to class 'Morning Yoga'.

Required equipment for yoga: mat

has_capacity (0/15 enrolled): True
is_suitable for beginner (fitness=1, difficulty=2): True
is_suitable for expert (fitness=5, difficulty=2): True

Overlaps with Sunday 10:00 (60 min): True
Overlaps with Sunday 11:00 (60 min): False
Overlaps with Monday 10:00 (60 min): False

Validation error: Difficulty level must be an integer between 1 and 5.
Validation error: Day must be one of ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'].


## 4. Trainer

**Design decisions:**
- Each trainer has a `max_weekly_classes` limit (default: 5). They can't be assigned more than this.
- `is_available(slot)` checks all assigned classes for time conflicts - used by both FitnessClass and PersonalTrainingSession.
- `analyze_training_statistics()` shows workload, class breakdown by category, total hours, and average class size. The assignment says we have creative freedom in what goes in this report.

In [7]:
class Trainer:
    """Represents a gym trainer who teaches fitness classes and personal training sessions."""

    _id_counter = 0

    def __init__(self, name, specialties, experience_years, max_weekly_classes=5):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Trainer name must be a non-empty string.")
        if not isinstance(specialties, list) or len(specialties) == 0:
            raise ValueError("Specialties must be a non-empty list.")
        if not isinstance(experience_years, int) or experience_years < 0:
            raise ValueError("Experience years must be a non-negative integer.")
        if not isinstance(max_weekly_classes, int) or max_weekly_classes < 1:
            raise ValueError("Max weekly classes must be a positive integer.")

        Trainer._id_counter += 1
        self._trainer_id = f"T{Trainer._id_counter:03d}"
        self._name = name
        self._specialties = [s.lower() for s in specialties]
        self._experience_years = experience_years
        self._max_weekly_classes = max_weekly_classes
        self._assigned_classes = []
        self._personal_sessions = []

    @property
    def trainer_id(self):
        return self._trainer_id

    @property
    def name(self):
        return self._name

    @property
    def specialties(self):
        return self._specialties

    @property
    def experience_years(self):
        return self._experience_years

    @property
    def max_weekly_classes(self):
        return self._max_weekly_classes

    @property
    def assigned_classes(self):
        return self._assigned_classes

    @property
    def personal_sessions(self):
        return self._personal_sessions

    def assign_class(self, fitness_class):
        """Take on a fitness class (checks weekly limit)."""
        if len(self._assigned_classes) >= self._max_weekly_classes:
            print(f"  Failed: {self._name} has reached the weekly limit of {self._max_weekly_classes} classes.")
            return False
        if fitness_class in self._assigned_classes:
            print(f"  Warning: {self._name} is already assigned to '{fitness_class.name}'.")
            return False
        if fitness_class.schedule_slot:
            if not self.is_available(fitness_class.schedule_slot, fitness_class.duration):
                print(f"  Failed: {self._name} has a schedule conflict at {fitness_class.schedule_slot}.")
                return False

        self._assigned_classes.append(fitness_class)
        fitness_class.assign_trainer(self)
        return True

    def remove_class(self, class_id):
        """Drop a class from this trainer's schedule."""
        for fc in self._assigned_classes:
            if fc.class_id == class_id:
                self._assigned_classes.remove(fc)
                print(f"  OK: {self._name} removed from class '{fc.name}'.")
                return True
        print(f"  Failed: {self._name} is not assigned to class '{class_id}'.")
        return False

    def is_available(self, slot, duration=60):
        """Check if the trainer is free at a given time slot."""
        for fc in self._assigned_classes:
            if fc.overlaps_with(slot, duration):
                return False
        for session in self._personal_sessions:
            if session.status == "scheduled" and session.is_conflict(slot, duration):
                return False
        return True

    def analyze_training_statistics(self):
        """Print a report on this trainer's workload and teaching statistics."""
        print(f"\n{'='*50}")
        print(f"  Trainer Statistics: {self._name} ({self._trainer_id})")
        print(f"{'='*50}")
        print(f"  Experience: {self._experience_years} years")
        print(f"  Specialties: {', '.join(self._specialties)}")
        print(f"  Classes assigned: {len(self._assigned_classes)}/{self._max_weekly_classes}")

        if self._assigned_classes:
            categories = {}
            total_enrolled = 0
            total_hours = 0
            for fc in self._assigned_classes:
                categories[fc.category] = categories.get(fc.category, 0) + 1
                total_enrolled += len(fc.enrolled_members)
                total_hours += fc.duration / 60

            print(f"  Classes by category:")
            for cat, count in categories.items():
                print(f"    - {cat}: {count}")

            avg_size = total_enrolled / len(self._assigned_classes)
            print(f"  Total teaching hours/week: {total_hours:.1f}")
            print(f"  Average class size: {avg_size:.1f}")
        else:
            print(f"  No classes assigned yet.")

        pt_count = len(self._personal_sessions)
        pt_completed = len([s for s in self._personal_sessions if s.status == "completed"])
        print(f"  Personal training sessions: {pt_count} (completed: {pt_completed})")
        print(f"{'='*50}")

    def __str__(self):
        return (f"Trainer({self._trainer_id}: {self._name}, "
                f"specialties={self._specialties}, "
                f"experience={self._experience_years}yr, "
                f"classes={len(self._assigned_classes)}/{self._max_weekly_classes})")

    def __eq__(self, other):
        if not isinstance(other, Trainer):
            return False
        return self._trainer_id == other._trainer_id

### Trainer — Usage Examples

In [8]:
# --- Trainer usage examples ---

trainer1 = Trainer("Dana Cohen", ["yoga", "pilates"], experience_years=5)
trainer2 = Trainer("Yossi Levi", ["spinning", "crossfit", "strength"], experience_years=8)
print(trainer1)
print(trainer2)

# Create and schedule a class, then assign a trainer
yoga_class = FitnessClass("Morning Yoga", "yoga", difficulty_level=2, capacity=15)
yoga_class.set_schedule(("Sunday", "10:00"))
trainer1.assign_class(yoga_class)
print(f"\n{trainer1}")

# Check availability
print(f"\nDana available Sunday 10:00? {trainer1.is_available(('Sunday', '10:00'))}")
print(f"Dana available Sunday 12:00? {trainer1.is_available(('Sunday', '12:00'))}")

# remove_class() - remove the class we just assigned
print("\n--- remove_class demo ---")
trainer1.remove_class(yoga_class.class_id)
print(f"Dana's classes after removal: {len(trainer1.assigned_classes)}")

# Statistics
trainer1.assign_class(yoga_class)  # re-assign for stats demo
trainer1.analyze_training_statistics()

# Validation
try:
    bad = Trainer("Test", [], 3)
except ValueError as e:
    print(f"\nValidation error: {e}")

Trainer._id_counter = 0
FitnessClass._id_counter = 0

Trainer(T001: Dana Cohen, specialties=['yoga', 'pilates'], experience=5yr, classes=0/5)
Trainer(T002: Yossi Levi, specialties=['spinning', 'crossfit', 'strength'], experience=8yr, classes=0/5)
  OK: Class 'Morning Yoga' scheduled for Sunday at 10:00.
  OK: Trainer 'Dana Cohen' assigned to class 'Morning Yoga'.

Trainer(T001: Dana Cohen, specialties=['yoga', 'pilates'], experience=5yr, classes=1/5)

Dana available Sunday 10:00? False
Dana available Sunday 12:00? True

--- remove_class demo ---
  OK: Dana Cohen removed from class 'Morning Yoga'.
Dana's classes after removal: 0
  OK: Trainer 'Dana Cohen' assigned to class 'Morning Yoga'.

  Trainer Statistics: Dana Cohen (T001)
  Experience: 5 years
  Specialties: yoga, pilates
  Classes assigned: 1/5
  Classes by category:
    - yoga: 1
  Total teaching hours/week: 1.0
  Average class size: 0.0
  Personal training sessions: 0 (completed: 0)

Validation error: Specialties must be a non-empty list.


## 5. Member

**Design decisions:**
- Members have a `fitness_level` (1-5), which determines which classes they can join.
- `can_attend(slot, duration)` checks against all current enrollments for time conflicts.
- `generate_payment_report()` delegates billing math to the `Subscription` class.
- `no_shows` is tracked as a simple counter. In a real system this would be per-class, but for this project a total count is cleaner.
- `request_personal_training()` creates a `PersonalTrainingSession` object and returns it.

In [9]:
class Member:
    """Represents a gym member with enrollments, payments, and personal training."""

    _id_counter = 0
    VALID_MEMBERSHIP_TYPES = ["monthly", "yearly", "premium"]

    def __init__(self, name, age, membership_type, fitness_level):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Member name must be a non-empty string.")
        if not isinstance(age, int) or age < 16 or age > 120:
            raise ValueError("Age must be an integer between 16 and 120.")
        if membership_type not in Member.VALID_MEMBERSHIP_TYPES:
            raise ValueError(f"Membership type must be one of {Member.VALID_MEMBERSHIP_TYPES}.")
        if not isinstance(fitness_level, int) or not 1 <= fitness_level <= 5:
            raise ValueError("Fitness level must be an integer between 1 and 5.")

        Member._id_counter += 1
        self._member_id = f"M{Member._id_counter:03d}"
        self._name = name
        self._age = age
        self._membership_type = membership_type
        self._fitness_level = fitness_level
        self._current_enrollments = []
        self._completed_classes = []
        self._no_shows = 0
        self._subscription = Subscription(self, membership_type)

    @property
    def member_id(self):
        return self._member_id

    @property
    def name(self):
        return self._name

    @property
    def age(self):
        return self._age

    @property
    def membership_type(self):
        return self._membership_type

    @property
    def fitness_level(self):
        return self._fitness_level

    @property
    def current_enrollments(self):
        return self._current_enrollments

    @property
    def completed_classes(self):
        return self._completed_classes

    @property
    def no_shows(self):
        return self._no_shows

    @property
    def subscription(self):
        return self._subscription

    def can_attend(self, slot, duration=60):
        """Check if the member is free at a given time slot."""
        for fc in self._current_enrollments:
            if fc.overlaps_with(slot, duration):
                return False
        return True

    def enroll_to_class(self, fitness_class):
        """Request to join a fitness class. The class handles all the checks."""
        return fitness_class.enroll(self)

    def cancel_enrollment(self, class_id):
        """Drop out of a fitness class by class ID."""
        for fc in self._current_enrollments:
            if fc.class_id == class_id:
                return fc.drop(self._member_id)
        print(f"  Failed: {self._name} is not enrolled in class '{class_id}'.")
        return False

    def add_completed_class(self, class_id):
        """Mark a class as completed."""
        self._completed_classes.append(class_id)
        print(f"  OK: {self._name} completed class '{class_id}'.")

    def add_no_show(self):
        """Record a no-show for this member."""
        self._no_shows += 1
        print(f"  Warning: No-show recorded for {self._name}. Total no-shows: {self._no_shows}")

    def request_personal_training(self, trainer, time_slot, duration=60, session_type="general"):
        """Book a 1-on-1 personal training session. Returns the session object."""
        session = PersonalTrainingSession(self, trainer, time_slot, duration, session_type)
        success = session.schedule()
        if success:
            return session
        return None

    def generate_payment_report(self):
        """Print the monthly payment report. Delegates billing math to Subscription."""
        classes_attended = len(self._completed_classes)
        bill = self._subscription.calculate_monthly_bill(classes_attended, self._no_shows)

        print(f"\n{'='*50}")
        print(f"  Payment Report: {self._name} ({self._member_id})")
        print(f"{'='*50}")
        print(f"  Plan: {self._membership_type}")
        print(f"  Base price: {self._subscription.monthly_price} NIS")
        print(f"  Classes attended: {classes_attended}")

        included = self._subscription.classes_included
        if included != float('inf'):
            extra = max(0, classes_attended - int(included))
            if extra > 0:
                print(f"  Extra classes ({extra} x {Subscription.EXTRA_CLASS_FEE} NIS): {extra * Subscription.EXTRA_CLASS_FEE} NIS")
        else:
            print(f"  Classes included: unlimited")

        if self._no_shows > 0:
            penalty = self._no_shows * Subscription.NO_SHOW_PENALTY
            print(f"  No-show penalties ({self._no_shows} x {Subscription.NO_SHOW_PENALTY} NIS): {penalty} NIS")

        print(f"  -------------------------")
        print(f"  Total due: {bill} NIS")
        print(f"{'='*50}")

    def __str__(self):
        return (f"Member({self._member_id}: {self._name}, age={self._age}, "
                f"plan={self._membership_type}, fitness={self._fitness_level}, "
                f"enrolled_in={len(self._current_enrollments)} classes)")

    def __eq__(self, other):
        if not isinstance(other, Member):
            return False
        return self._member_id == other._member_id

### Member — Usage Examples

Note: `request_personal_training()` will be demoed after we define `PersonalTrainingSession` below.

In [10]:
# --- Member usage examples ---

m1 = Member("Ido Malach", 26, "monthly", fitness_level=3)
m2 = Member("Yonatan Dolman", 24, "premium", fitness_level=1)
print(m1)
print(m2)

# Create a class with a trainer and schedule
t1 = Trainer("Dana Cohen", ["yoga", "pilates"], experience_years=5)
yoga = FitnessClass("Morning Yoga", "yoga", difficulty_level=2, capacity=2)
yoga.set_schedule(("Sunday", "10:00"))
t1.assign_class(yoga)

# Enrollment tests
print("\n--- Enrollment tests ---")
m1.enroll_to_class(yoga)
m2.enroll_to_class(yoga)  # fitness=1, difficulty=2: 2 <= 1+1 = OK

# can_attend() - check if m1 is free at a given time
print("\n--- can_attend demo ---")
print(f"m1 can attend Sunday 10:00? {m1.can_attend(('Sunday', '10:00'))}")   # False - already in yoga
print(f"m1 can attend Monday 10:00? {m1.can_attend(('Monday', '10:00'))}")   # True - free

# 3rd member for waitlist
m3 = Member("Etan Cohen", 25, "yearly", fitness_level=4)
m3.enroll_to_class(yoga)  # class full -> waitlist

# Drop m1 -> m3 promoted
print("\n--- Drop and promote ---")
yoga.drop(m1.member_id)

print(f"\nYoga enrolled: {[m.name for m in yoga.enrolled_members]}")
print(f"Yoga waiting list: {[m.name for m in yoga.waiting_list]}")

# Fitness mismatch
print("\n--- Fitness mismatch ---")
hard_class = FitnessClass("Advanced CrossFit", "crossfit", difficulty_level=5, capacity=10)
hard_class.set_schedule(("Monday", "08:00"))
m2.enroll_to_class(hard_class)  # 5 > 1+1 -> REJECTED

# Payment report
print("\n--- Payment report ---")
m1.add_completed_class("C001")
m1.add_completed_class("C002")
m1.add_no_show()
m1.generate_payment_report()

# Validation
try:
    bad = Member("Test", 15, "monthly", 3)
except ValueError as e:
    print(f"\nValidation error: {e}")

Member._id_counter = 0
Trainer._id_counter = 0
FitnessClass._id_counter = 0

Member(M001: Ido Malach, age=26, plan=monthly, fitness=3, enrolled_in=0 classes)
Member(M002: Yonatan Dolman, age=24, plan=premium, fitness=1, enrolled_in=0 classes)
  OK: Class 'Morning Yoga' scheduled for Sunday at 10:00.
  OK: Trainer 'Dana Cohen' assigned to class 'Morning Yoga'.

--- Enrollment tests ---
  OK: Ido Malach enrolled in 'Morning Yoga'.
  OK: Yonatan Dolman enrolled in 'Morning Yoga'.

--- can_attend demo ---
m1 can attend Sunday 10:00? False
m1 can attend Monday 10:00? True
  FULL: Class 'Morning Yoga' is full. Adding Etan Cohen to waiting list.
  OK: Etan Cohen added to waiting list for 'Morning Yoga'.

--- Drop and promote ---
  OK: Ido Malach dropped from 'Morning Yoga'.
  PROMOTED: Etan Cohen promoted from waiting list to 'Morning Yoga'.

Yoga enrolled: ['Yonatan Dolman', 'Etan Cohen']
Yoga waiting list: []

--- Fitness mismatch ---
  OK: Class 'Advanced CrossFit' scheduled for Monday at 08:00.
  REJECTED: Yonatan Dolman (fitness level 1) cannot join 'Advanced Cro

## 6. PersonalTrainingSession

**Why this extra class?** The assignment already has `Member.request_personal_training()` and `Trainer.is_available()` — these methods need a class to create and manage. This models a real 1-on-1 session with scheduling, cancellation, completion, rescheduling, and conflict detection.

**Design decisions:**
- Status can be: `"scheduled"`, `"completed"`, or `"cancelled"`.
- `schedule()` validates both trainer availability and member schedule conflicts before booking.
- `complete(notes)` allows the trainer to attach notes after the session.
- The session is added to `trainer.personal_sessions` and tracked for conflict detection.

In [11]:
class PersonalTrainingSession:
    """Represents a 1-on-1 personal training session between a member and a trainer."""

    _id_counter = 0
    VALID_STATUSES = ["scheduled", "completed", "cancelled"]

    def __init__(self, member, trainer, time_slot, duration=60, session_type="general"):
        if not isinstance(member, Member):
            raise TypeError("member must be a Member object.")
        if not isinstance(trainer, Trainer):
            raise TypeError("trainer must be a Trainer object.")
        if not isinstance(time_slot, tuple) or len(time_slot) != 2:
            raise ValueError("Time slot must be a tuple of (day, time).")
        if not isinstance(duration, int) or duration < 1:
            raise ValueError("Duration must be a positive integer (minutes).")

        PersonalTrainingSession._id_counter += 1
        self._session_id = f"S{PersonalTrainingSession._id_counter:03d}"
        self._member = member
        self._trainer = trainer
        self._time_slot = time_slot
        self._duration = duration
        self._session_type = session_type
        self._status = "scheduled"
        self._notes = ""

    @property
    def session_id(self):
        return self._session_id

    @property
    def member(self):
        return self._member

    @property
    def trainer(self):
        return self._trainer

    @property
    def time_slot(self):
        return self._time_slot

    @property
    def duration(self):
        return self._duration

    @property
    def session_type(self):
        return self._session_type

    @property
    def status(self):
        return self._status

    @property
    def notes(self):
        return self._notes

    def is_conflict(self, other_slot, other_duration=60):
        """Check if a given time slot overlaps with this session."""
        if self._time_slot[0] != other_slot[0]:
            return False
        my_start = int(self._time_slot[1].split(":")[0]) * 60 + int(self._time_slot[1].split(":")[1])
        my_end = my_start + self._duration
        other_start = int(other_slot[1].split(":")[0]) * 60 + int(other_slot[1].split(":")[1])
        other_end = other_start + other_duration
        return my_start < other_end and other_start < my_end

    def schedule(self):
        """Validate trainer availability and member conflicts, then book."""
        if not self._trainer.is_available(self._time_slot, self._duration):
            print(f"  Failed: Trainer {self._trainer.name} is not available at {self._time_slot}.")
            self._status = "cancelled"
            return False

        if not self._member.can_attend(self._time_slot, self._duration):
            print(f"  Failed: {self._member.name} has a schedule conflict at {self._time_slot}.")
            self._status = "cancelled"
            return False

        self._trainer.personal_sessions.append(self)
        self._status = "scheduled"
        print(f"  OK: Personal training session {self._session_id} booked: "
              f"{self._member.name} with {self._trainer.name} on "
              f"{self._time_slot[0]} at {self._time_slot[1]} ({self._session_type}).")
        return True

    def cancel(self):
        """Cancel the session."""
        if self._status == "cancelled":
            print(f"  Session {self._session_id} is already cancelled.")
            return
        if self._status == "completed":
            print(f"  Session {self._session_id} is already completed - cannot cancel.")
            return
        self._status = "cancelled"
        print(f"  OK: Session {self._session_id} cancelled "
              f"({self._member.name} with {self._trainer.name}).")

    def complete(self, notes=""):
        """Mark the session as completed and save trainer notes."""
        if self._status != "scheduled":
            print(f"  Failed: Cannot complete session {self._session_id} (status: {self._status}).")
            return
        self._status = "completed"
        self._notes = notes
        print(f"  OK: Session {self._session_id} completed. Notes: {notes if notes else '(none)'}")

    def reschedule(self, new_slot):
        """Move the session to a different time slot (re-validates conflicts)."""
        if self._status != "scheduled":
            print(f"  Failed: Cannot reschedule session {self._session_id} (status: {self._status}).")
            return False

        if self in self._trainer.personal_sessions:
            self._trainer.personal_sessions.remove(self)

        if not self._trainer.is_available(new_slot, self._duration):
            print(f"  Failed: Trainer {self._trainer.name} is not available at {new_slot}.")
            self._trainer.personal_sessions.append(self)
            return False

        if not self._member.can_attend(new_slot, self._duration):
            print(f"  Failed: {self._member.name} has a conflict at {new_slot}.")
            self._trainer.personal_sessions.append(self)
            return False

        old_slot = self._time_slot
        self._time_slot = new_slot
        self._trainer.personal_sessions.append(self)
        print(f"  OK: Session {self._session_id} rescheduled from "
              f"{old_slot[0]} {old_slot[1]} to {new_slot[0]} {new_slot[1]}.")
        return True

    def __str__(self):
        return (f"PersonalTrainingSession({self._session_id}: "
                f"{self._member.name} with {self._trainer.name}, "
                f"{self._time_slot[0]} {self._time_slot[1]}, "
                f"type={self._session_type}, status={self._status})")

    def __eq__(self, other):
        if not isinstance(other, PersonalTrainingSession):
            return False
        return self._session_id == other._session_id

### PersonalTrainingSession — Usage Examples

In [12]:
# --- PersonalTrainingSession usage examples ---

t1 = Trainer("Dana Cohen", ["yoga", "pilates"], experience_years=5)
m1 = Member("Ido Malach", 26, "monthly", fitness_level=3)

# Book via Member.request_personal_training()
print("--- Booking a session ---")
session = m1.request_personal_training(t1, ("Wednesday", "14:00"), duration=60, session_type="strength")
if session:
    print(session)

# is_conflict() - check if a time slot overlaps with this session
print("\n--- is_conflict demo ---")
print(f"Conflicts with Wednesday 14:00? {session.is_conflict(('Wednesday', '14:00'))}")   # True
print(f"Conflicts with Wednesday 15:00? {session.is_conflict(('Wednesday', '15:00'))}")   # False
print(f"Conflicts with Thursday 14:00? {session.is_conflict(('Thursday', '14:00'))}")     # False

# Conflict test - another member tries same trainer/time
print("\n--- Booking conflict test ---")
m2 = Member("Yonatan Dolman", 24, "premium", fitness_level=2)
session2 = m2.request_personal_training(t1, ("Wednesday", "14:00"))

# Reschedule
print("\n--- Reschedule ---")
session.reschedule(("Wednesday", "16:00"))
print(session)

# Complete
print("\n--- Complete ---")
session.complete("Good progress on deadlifts. Increase weight next session.")
print(session)

# Try to cancel completed
print("\n--- Cancel completed ---")
session.cancel()

Trainer._id_counter = 0
Member._id_counter = 0
PersonalTrainingSession._id_counter = 0

--- Booking a session ---
  OK: Personal training session S001 booked: Ido Malach with Dana Cohen on Wednesday at 14:00 (strength).
PersonalTrainingSession(S001: Ido Malach with Dana Cohen, Wednesday 14:00, type=strength, status=scheduled)

--- is_conflict demo ---
Conflicts with Wednesday 14:00? True
Conflicts with Wednesday 15:00? False
Conflicts with Thursday 14:00? False

--- Booking conflict test ---
  Failed: Trainer Dana Cohen is not available at ('Wednesday', '14:00').

--- Reschedule ---
  OK: Session S001 rescheduled from Wednesday 14:00 to Wednesday 16:00.
PersonalTrainingSession(S001: Ido Malach with Dana Cohen, Wednesday 16:00, type=strength, status=scheduled)

--- Complete ---
  OK: Session S001 completed. Notes: Good progress on deadlifts. Increase weight next session.
PersonalTrainingSession(S001: Ido Malach with Dana Cohen, Wednesday 16:00, type=strength, status=completed)

--- Cancel completed ---
  Session S001 is already completed - cannot cancel.


## 7. Gym

The top-level container class. Everything comes together here — all members, trainers, classes, and equipment.

**Reports:**
- `generate_schedule_report()` — day-by-day weekly view with conflict flags (trainer double-booking, time overlaps, equipment issues).
- `generate_membership_report()` — breakdown by plan type, fitness level distribution, active subscriptions.
- `search_classes(search_term)` — simple string matching across class name, category, difficulty, and trainer name.

In [13]:
class Gym:
    """Top-level class that manages all gym operations."""

    def __init__(self, name):
        if not isinstance(name, str) or not name.strip():
            raise ValueError("Gym name must be a non-empty string.")
        self._name = name
        self._members = []
        self._trainers = []
        self._classes = []
        self._equipment_inventory = []

    @property
    def name(self):
        return self._name

    @property
    def members(self):
        return self._members

    @property
    def trainers(self):
        return self._trainers

    @property
    def classes(self):
        return self._classes

    @property
    def equipment_inventory(self):
        return self._equipment_inventory

    def add_member(self, member):
        """Register a member to the gym."""
        if not isinstance(member, Member):
            raise TypeError("Must be a Member object.")
        if member in self._members:
            print(f"  Warning: {member.name} is already a member of {self._name}.")
            return False
        self._members.append(member)
        print(f"  OK: {member.name} registered at {self._name}.")
        return True

    def add_trainer(self, trainer):
        """Add a trainer to the gym."""
        if not isinstance(trainer, Trainer):
            raise TypeError("Must be a Trainer object.")
        if trainer in self._trainers:
            print(f"  Warning: {trainer.name} is already a trainer at {self._name}.")
            return False
        self._trainers.append(trainer)
        print(f"  OK: Trainer {trainer.name} added to {self._name}.")
        return True

    def add_class(self, fitness_class):
        """Add a fitness class to the gym's offerings."""
        if not isinstance(fitness_class, FitnessClass):
            raise TypeError("Must be a FitnessClass object.")
        if fitness_class in self._classes:
            print(f"  Warning: Class '{fitness_class.name}' already exists at {self._name}.")
            return False
        self._classes.append(fitness_class)
        print(f"  OK: Class '{fitness_class.name}' added to {self._name}.")
        return True

    def add_equipment(self, equipment):
        """Add equipment to the gym's inventory."""
        if not isinstance(equipment, Equipment):
            raise TypeError("Must be an Equipment object.")
        if equipment in self._equipment_inventory:
            print(f"  Warning: {equipment.name} is already in inventory.")
            return False
        self._equipment_inventory.append(equipment)
        print(f"  OK: {equipment.name} added to {self._name} inventory.")
        return True

    def search_classes(self, search_term):
        """Find classes by name, category, difficulty, or trainer name."""
        search_term = str(search_term).lower()
        results = []
        for fc in self._classes:
            if (search_term in fc.name.lower()
                or search_term in fc.category.lower()
                or search_term == str(fc.difficulty_level)
                or (fc.trainer and search_term in fc.trainer.name.lower())):
                results.append(fc)

        if results:
            print(f"\n  Search results for '{search_term}':")
            for fc in results:
                print(f"    - {fc}")
        else:
            print(f"\n  No classes found matching '{search_term}'.")
        return results

    def generate_schedule_report(self):
        """Print a full weekly schedule with conflict/overlap warnings."""
        print(f"\n{'='*65}")
        print(f"  Weekly Schedule Report - {self._name}")
        print(f"{'='*65}")

        days = ["Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
        conflicts = []

        for day in days:
            day_classes = [fc for fc in self._classes
                          if fc.schedule_slot and fc.schedule_slot[0] == day]
            day_classes.sort(key=lambda fc: fc.schedule_slot[1])

            if day_classes:
                print(f"\n  {day}:")
                for fc in day_classes:
                    trainer_name = fc.trainer.name if fc.trainer else "No trainer"
                    end_time = fc._get_end_time()
                    print(f"    {fc.schedule_slot[1]}-{end_time}  "
                          f"{fc.name} | Trainer: {trainer_name} | "
                          f"Enrolled: {len(fc.enrolled_members)}/{fc.capacity}")

            for i in range(len(day_classes)):
                for j in range(i + 1, len(day_classes)):
                    fc1 = day_classes[i]
                    fc2 = day_classes[j]
                    if fc1.overlaps_with(fc2.schedule_slot, fc2.duration):
                        conflicts.append(f"TIME OVERLAP: '{fc1.name}' and '{fc2.name}' on {day}")
                    if (fc1.trainer and fc2.trainer
                        and fc1.trainer == fc2.trainer
                        and fc1.overlaps_with(fc2.schedule_slot, fc2.duration)):
                        conflicts.append(f"TRAINER CONFLICT: {fc1.trainer.name} double-booked on {day}")

        for eq in self._equipment_inventory:
            if eq.condition == "needs_repair" and eq.assigned_to_classes:
                for fc in eq.assigned_to_classes:
                    conflicts.append(f"EQUIPMENT: '{eq.name}' needs repair - affects '{fc.name}'")

        print(f"\n  " + "-"*50)
        if conflicts:
            print(f"  WARNINGS ({len(conflicts)}):")
            for c in conflicts:
                print(f"    - {c}")
        else:
            print(f"  No scheduling conflicts detected.")
        print(f"{'='*65}")

    def generate_membership_report(self):
        """Print a breakdown of membership types and fitness levels."""
        print(f"\n{'='*50}")
        print(f"  Membership Report - {self._name}")
        print(f"{'='*50}")
        print(f"  Total members: {len(self._members)}")

        plan_counts = {}
        for m in self._members:
            plan_counts[m.membership_type] = plan_counts.get(m.membership_type, 0) + 1
        print(f"\n  By plan type:")
        for plan, count in plan_counts.items():
            print(f"    - {plan}: {count}")

        level_counts = {}
        for m in self._members:
            level_counts[m.fitness_level] = level_counts.get(m.fitness_level, 0) + 1
        print(f"\n  By fitness level:")
        for level in sorted(level_counts.keys()):
            print(f"    - Level {level}: {level_counts[level]}")

        active = len([m for m in self._members if m.subscription.is_active])
        inactive = len(self._members) - active
        print(f"\n  Active subscriptions: {active}")
        print(f"  Inactive subscriptions: {inactive}")
        print(f"{'='*50}")

    def __str__(self):
        return (f"Gym('{self._name}': {len(self._members)} members, "
                f"{len(self._trainers)} trainers, {len(self._classes)} classes, "
                f"{len(self._equipment_inventory)} equipment)")

### Gym — Usage Examples

Basic creation and add operations. The full demonstration of all reports happens in the Simulation section below.

In [14]:
# --- Gym usage examples ---

gym = Gym("FitZone BGU")
print(gym)

t = Trainer("Dana Cohen", ["yoga"], experience_years=5)
gym.add_trainer(t)

c = FitnessClass("Yoga Flow", "yoga", difficulty_level=2, capacity=10)
c.set_schedule(("Sunday", "09:00"))
t.assign_class(c)
gym.add_class(c)

m = Member("Test User", 25, "monthly", fitness_level=3)
gym.add_member(m)
m.enroll_to_class(c)

eq = Equipment("Yoga Mats", "mat", 20)
gym.add_equipment(eq)
eq.reserve(c)

print(f"\n{gym}")

# Search
gym.search_classes("yoga")
gym.search_classes("dana")
gym.search_classes("boxing")

# Validation
try:
    gym.add_member("not a member")
except TypeError as e:
    print(f"\nValidation error: {e}")

# Reset all counters for the simulation section
Equipment._id_counter = 0
FitnessClass._id_counter = 0
Trainer._id_counter = 0
Member._id_counter = 0
PersonalTrainingSession._id_counter = 0

Gym('FitZone BGU': 0 members, 0 trainers, 0 classes, 0 equipment)
  OK: Trainer Dana Cohen added to FitZone BGU.
  OK: Class 'Yoga Flow' scheduled for Sunday at 09:00.
  OK: Trainer 'Dana Cohen' assigned to class 'Yoga Flow'.
  OK: Class 'Yoga Flow' added to FitZone BGU.
  OK: Test User registered at FitZone BGU.
  OK: Test User enrolled in 'Yoga Flow'.
  OK: Yoga Mats added to FitZone BGU inventory.
  OK: Yoga Mats reserved for class 'Yoga Flow'.

Gym('FitZone BGU': 1 members, 1 trainers, 1 classes, 1 equipment)

  Search results for 'yoga':
    - FitnessClass(C001: 'Yoga Flow', category=yoga, difficulty=2, trainer=Dana Cohen, schedule=Sunday 09:00, enrolled=1/10)

  Search results for 'dana':
    - FitnessClass(C001: 'Yoga Flow', category=yoga, difficulty=2, trainer=Dana Cohen, schedule=Sunday 09:00, enrolled=1/10)

  No classes found matching 'boxing'.

Validation error: Must be a Member object.


---

# Full Simulation

This section creates a complete gym scenario from scratch and demonstrates all the system's features working together. The simulation is **deterministic** — no randomness is involved. This is an OOP project, not a probability project (yet).

However, the `Equipment` class is architecturally ready for probabilistic breakdowns — for example, each piece of equipment could have a random chance of malfunctioning based on usage. We could add this in a future extension using the sampling techniques from weeks 3-4.

**Scenario:** 3 trainers, 7 members, 5 classes, 4 equipment types.

### Step 1: Create the Gym and add Trainers

In [15]:
# ============================================================
# STEP 1: Create the gym and add trainers
# ============================================================

gym = Gym("FitZone Beer-Sheva")

# 3 trainers with different specialties
dana = Trainer("Dana Cohen", ["yoga", "pilates"], experience_years=5)
yossi = Trainer("Yossi Levi", ["spinning", "crossfit", "strength"], experience_years=8)
noa = Trainer("Noa Shapira", ["boxing", "crossfit"], experience_years=3)

gym.add_trainer(dana)
gym.add_trainer(yossi)
gym.add_trainer(noa)

print(f"\n{gym}")

  OK: Trainer Dana Cohen added to FitZone Beer-Sheva.
  OK: Trainer Yossi Levi added to FitZone Beer-Sheva.
  OK: Trainer Noa Shapira added to FitZone Beer-Sheva.

Gym('FitZone Beer-Sheva': 0 members, 3 trainers, 0 classes, 0 equipment)


### Step 2: Add Equipment

In [16]:
# ============================================================
# STEP 2: Add equipment to inventory
# ============================================================

bikes = Equipment("Spinning Bikes", "bike", 15)
mats = Equipment("Yoga Mats", "mat", 25)
weights = Equipment("Free Weights Set", "weights", 10)
bags = Equipment("Punching Bags", "punching_bag", 8)

gym.add_equipment(bikes)
gym.add_equipment(mats)
gym.add_equipment(weights)
gym.add_equipment(bags)

print(f"\n{gym}")

  OK: Spinning Bikes added to FitZone Beer-Sheva inventory.
  OK: Yoga Mats added to FitZone Beer-Sheva inventory.
  OK: Free Weights Set added to FitZone Beer-Sheva inventory.
  OK: Punching Bags added to FitZone Beer-Sheva inventory.

Gym('FitZone Beer-Sheva': 0 members, 3 trainers, 0 classes, 4 equipment)


### Step 3: Create Fitness Classes with schedules, trainers, and equipment

In [17]:
# ============================================================
# STEP 3: Create classes, set schedules, assign trainers + equipment
# ============================================================

# Class 1: Morning Yoga - easy, big capacity
morning_yoga = FitnessClass("Morning Yoga", "yoga", difficulty_level=2, capacity=20)
morning_yoga.set_schedule(("Sunday", "08:00"))
dana.assign_class(morning_yoga)
mats.reserve(morning_yoga)
gym.add_class(morning_yoga)

# Class 2: Power Spinning - moderate difficulty
power_spin = FitnessClass("Power Spinning", "spinning", difficulty_level=3, capacity=12)
power_spin.set_schedule(("Sunday", "10:00"))
yossi.assign_class(power_spin)
bikes.reserve(power_spin)
gym.add_class(power_spin)

# Class 3: Pilates Core - moderate difficulty
pilates = FitnessClass("Pilates Core", "pilates", difficulty_level=3, capacity=15)
pilates.set_schedule(("Monday", "09:00"))
dana.assign_class(pilates)
mats.reserve(pilates)
gym.add_class(pilates)

# Class 4: CrossFit Blast - hard, small capacity (will trigger waitlist)
crossfit = FitnessClass("CrossFit Blast", "crossfit", difficulty_level=4, capacity=3)
crossfit.set_schedule(("Tuesday", "17:00"))
yossi.assign_class(crossfit)
weights.reserve(crossfit)
gym.add_class(crossfit)

# Class 5: Boxing Fundamentals - moderate
boxing = FitnessClass("Boxing Fundamentals", "boxing", difficulty_level=3, capacity=10)
boxing.set_schedule(("Wednesday", "18:00"))
noa.assign_class(boxing)
bags.reserve(boxing)
gym.add_class(boxing)

print(f"\n{gym}")

  OK: Class 'Morning Yoga' scheduled for Sunday at 08:00.
  OK: Trainer 'Dana Cohen' assigned to class 'Morning Yoga'.
  OK: Yoga Mats reserved for class 'Morning Yoga'.
  OK: Class 'Morning Yoga' added to FitZone Beer-Sheva.
  OK: Class 'Power Spinning' scheduled for Sunday at 10:00.
  OK: Trainer 'Yossi Levi' assigned to class 'Power Spinning'.
  OK: Spinning Bikes reserved for class 'Power Spinning'.
  OK: Class 'Power Spinning' added to FitZone Beer-Sheva.
  OK: Class 'Pilates Core' scheduled for Monday at 09:00.
  OK: Trainer 'Dana Cohen' assigned to class 'Pilates Core'.
  OK: Yoga Mats reserved for class 'Pilates Core'.
  OK: Class 'Pilates Core' added to FitZone Beer-Sheva.
  OK: Class 'CrossFit Blast' scheduled for Tuesday at 17:00.
  OK: Trainer 'Yossi Levi' assigned to class 'CrossFit Blast'.
  OK: Free Weights Set reserved for class 'CrossFit Blast'.
  OK: Class 'CrossFit Blast' added to FitZone Beer-Sheva.
  OK: Class 'Boxing Fundamentals' scheduled for Wednesday at 18:00.

### Step 4: Add Members with different plans and fitness levels

In [18]:
# ============================================================
# STEP 4: Add members - mix of plans and fitness levels
# ============================================================

# Monthly plan members
ido = Member("Ido Malach", 26, "monthly", fitness_level=3)
yonatan = Member("Yonatan Dolman", 24, "monthly", fitness_level=4)

# Yearly plan members
eitan = Member("Etan Cohen", 25, "yearly", fitness_level=2)
maya = Member("Maya Rosenberg", 30, "yearly", fitness_level=5)

# Premium plan members
tom = Member("Tom Alon", 22, "premium", fitness_level=3)
shira = Member("Shira Dayan", 28, "premium", fitness_level=1)

# One more monthly (will test fitness mismatch)
amir = Member("Amir Katz", 35, "monthly", fitness_level=1)

for member in [ido, yonatan, eitan, maya, tom, shira, amir]:
    gym.add_member(member)

print(f"\n{gym}")

  OK: Ido Malach registered at FitZone Beer-Sheva.
  OK: Yonatan Dolman registered at FitZone Beer-Sheva.
  OK: Etan Cohen registered at FitZone Beer-Sheva.
  OK: Maya Rosenberg registered at FitZone Beer-Sheva.
  OK: Tom Alon registered at FitZone Beer-Sheva.
  OK: Shira Dayan registered at FitZone Beer-Sheva.
  OK: Amir Katz registered at FitZone Beer-Sheva.

Gym('FitZone Beer-Sheva': 7 members, 3 trainers, 5 classes, 4 equipment)


### Step 5: Enrollments — successful, waitlist, and rejection demos

In [19]:
# ============================================================
# STEP 5: Enrollments - success, waitlist, and rejection
# ============================================================

print("=== Successful enrollments ===")
# Morning Yoga (difficulty 2) - everyone with fitness >= 1 can join
ido.enroll_to_class(morning_yoga)
eitan.enroll_to_class(morning_yoga)
shira.enroll_to_class(morning_yoga)

# Power Spinning (difficulty 3) - need fitness >= 2
ido.enroll_to_class(power_spin)
yonatan.enroll_to_class(power_spin)
tom.enroll_to_class(power_spin)

# Pilates Core (difficulty 3)
maya.enroll_to_class(pilates)
eitan.enroll_to_class(pilates)  # fitness 2, difficulty 3: 3 <= 2+1 = OK

# Boxing Fundamentals (difficulty 3)
yonatan.enroll_to_class(boxing)
tom.enroll_to_class(boxing)
maya.enroll_to_class(boxing)

print("\n=== CrossFit Blast (capacity=3) - testing waitlist ===")
# CrossFit Blast (difficulty 4, capacity 3) - need fitness >= 3
ido.enroll_to_class(crossfit)       # fills spot 1
yonatan.enroll_to_class(crossfit)   # fills spot 2
maya.enroll_to_class(crossfit)      # fills spot 3 (class full)
tom.enroll_to_class(crossfit)       # suitable but full -> waitlist

print("\n=== Fitness mismatch rejection ===")
# Amir (fitness=1) tries CrossFit Blast (difficulty=4): 4 > 1+1 -> REJECTED
amir.enroll_to_class(crossfit)

# Shira (fitness=1) tries Power Spinning (difficulty=3): 3 > 1+1 -> REJECTED
shira.enroll_to_class(power_spin)

=== Successful enrollments ===
  OK: Ido Malach enrolled in 'Morning Yoga'.
  OK: Etan Cohen enrolled in 'Morning Yoga'.
  OK: Shira Dayan enrolled in 'Morning Yoga'.
  OK: Ido Malach enrolled in 'Power Spinning'.
  OK: Yonatan Dolman enrolled in 'Power Spinning'.
  OK: Tom Alon enrolled in 'Power Spinning'.
  OK: Maya Rosenberg enrolled in 'Pilates Core'.
  OK: Etan Cohen enrolled in 'Pilates Core'.
  OK: Yonatan Dolman enrolled in 'Boxing Fundamentals'.
  OK: Tom Alon enrolled in 'Boxing Fundamentals'.
  OK: Maya Rosenberg enrolled in 'Boxing Fundamentals'.

=== CrossFit Blast (capacity=3) - testing waitlist ===
  OK: Ido Malach enrolled in 'CrossFit Blast'.
  OK: Yonatan Dolman enrolled in 'CrossFit Blast'.
  OK: Maya Rosenberg enrolled in 'CrossFit Blast'.
  FULL: Class 'CrossFit Blast' is full. Adding Tom Alon to waiting list.
  OK: Tom Alon added to waiting list for 'CrossFit Blast'.

=== Fitness mismatch rejection ===
  REJECTED: Amir Katz (fitness level 1) cannot join 'CrossFit

False

### Step 6: Waitlist promotion — drop a member and watch auto-promotion

In [20]:
# ============================================================
# STEP 6: Drop a member from CrossFit -> Tom gets promoted from waitlist
# ============================================================

print(f"CrossFit before drop: {[m.name for m in crossfit.enrolled_members]}")
print(f"Waitlist: {[m.name for m in crossfit.waiting_list]}")

# Ido drops out of CrossFit
ido.cancel_enrollment(crossfit.class_id)

print(f"\nCrossFit after drop: {[m.name for m in crossfit.enrolled_members]}")
print(f"Waitlist: {[m.name for m in crossfit.waiting_list]}")

CrossFit before drop: ['Ido Malach', 'Yonatan Dolman', 'Maya Rosenberg']
Waitlist: ['Tom Alon']
  OK: Ido Malach dropped from 'CrossFit Blast'.
  PROMOTED: Tom Alon promoted from waiting list to 'CrossFit Blast'.

CrossFit after drop: ['Yonatan Dolman', 'Maya Rosenberg', 'Tom Alon']
Waitlist: []


### Step 7: Book a Personal Training Session

In [21]:
# ============================================================
# STEP 7: Personal training sessions + Trainer.remove_class() demo
# ============================================================

# First, demo remove_class: Noa drops Boxing, then re-assigns it
print("--- Trainer.remove_class demo ---")
noa.remove_class(boxing.class_id)
print(f"Noa's classes after removal: {len(noa.assigned_classes)}")
noa.assign_class(boxing)  # re-assign
print(f"Noa's classes after re-assign: {len(noa.assigned_classes)}")

# Ido books a strength session with Yossi on Thursday
print("\n--- Booking personal training ---")
pt_session = ido.request_personal_training(
    yossi, ("Thursday", "14:00"), duration=60, session_type="strength"
)

# Maya books a session with Dana on Wednesday
pt_session2 = maya.request_personal_training(
    dana, ("Wednesday", "11:00"), duration=60, session_type="flexibility"
)

# Tom tries to book with Yossi at the same time as Ido - should fail
print("\n--- Conflict test ---")
pt_fail = tom.request_personal_training(
    yossi, ("Thursday", "14:00"), duration=60, session_type="cardio"
)

# Complete Ido's session
print("\n--- Complete session ---")
pt_session.complete("Great session. Focus on form for squats next time.")

--- Trainer.remove_class demo ---
  OK: Noa Shapira removed from class 'Boxing Fundamentals'.
Noa's classes after removal: 0
  OK: Trainer 'Noa Shapira' assigned to class 'Boxing Fundamentals'.
Noa's classes after re-assign: 1

--- Booking personal training ---
  OK: Personal training session S001 booked: Ido Malach with Yossi Levi on Thursday at 14:00 (strength).
  OK: Personal training session S002 booked: Maya Rosenberg with Dana Cohen on Wednesday at 11:00 (flexibility).

--- Conflict test ---
  Failed: Trainer Yossi Levi is not available at ('Thursday', '14:00').

--- Complete session ---
  OK: Session S001 completed. Notes: Great session. Focus on form for squats next time.


### Step 8: Equipment Malfunction — cascade warning

In [22]:
# ============================================================
# STEP 8: Equipment malfunction -> warning about affected classes
# ============================================================

# The mats break down - this affects Morning Yoga and Pilates Core
mats.report_malfunction()

mats.get_usage_report()

# Repair the mats
print("\n--- Repairing ---")
mats.repair()
mats.get_usage_report()

# Demo release: remove mats from Morning Yoga, then re-reserve
print("\n--- Equipment release and re-reserve demo ---")
mats.release(morning_yoga)
print(f"Mats assigned to {len(mats.assigned_to_classes)} class(es) after release")
mats.reserve(morning_yoga)
print(f"Mats assigned to {len(mats.assigned_to_classes)} class(es) after re-reserve")


MALFUNCTION REPORTED: Yoga Mats (E002)
  WARNING - The following classes are affected:
    - Morning Yoga (ID: C001)
    - Pilates Core (ID: C003)

--- Equipment Usage Report: Yoga Mats (E002) ---
  Type: mat
  Condition: needs_repair
  Quantity available: 25
  Assigned to 2 class(es):
    - Morning Yoga
    - Pilates Core

--- Repairing ---
  OK: Yoga Mats has been repaired and is back in service.

--- Equipment Usage Report: Yoga Mats (E002) ---
  Type: mat
  Condition: good
  Quantity available: 25
  Assigned to 2 class(es):
    - Morning Yoga
    - Pilates Core

--- Equipment release and re-reserve demo ---
  OK: Yoga Mats released from class 'Morning Yoga'.
Mats assigned to 1 class(es) after release
  OK: Yoga Mats reserved for class 'Morning Yoga'.
Mats assigned to 2 class(es) after re-reserve


### Step 9: Simulate completed classes and no-shows for payment reports

In [23]:
# ============================================================
# STEP 9: Simulate some completed classes and no-shows
# ============================================================

print("--- Simulating class completions and no-shows ---")

# Ido attended yoga and spinning, missed one class (no-show)
ido.add_completed_class(morning_yoga.class_id)
ido.add_completed_class(power_spin.class_id)
ido.add_no_show()

# Yonatan attended 3 classes
yonatan.add_completed_class(power_spin.class_id)
yonatan.add_completed_class(crossfit.class_id)
yonatan.add_completed_class(boxing.class_id)

# Maya attended 4 classes (yearly plan, 12 included)
maya.add_completed_class(pilates.class_id)
maya.add_completed_class(crossfit.class_id)
maya.add_completed_class(boxing.class_id)
maya.add_completed_class(morning_yoga.class_id)

# Tom attended 2 classes (premium, unlimited)
tom.add_completed_class(power_spin.class_id)
tom.add_completed_class(boxing.class_id)

# Shira attended 1 class, 1 no-show
shira.add_completed_class(morning_yoga.class_id)
shira.add_no_show()

--- Simulating class completions and no-shows ---
  OK: Ido Malach completed class 'C001'.
  OK: Ido Malach completed class 'C002'.
  OK: Yonatan Dolman completed class 'C002'.
  OK: Yonatan Dolman completed class 'C004'.
  OK: Yonatan Dolman completed class 'C005'.
  OK: Maya Rosenberg completed class 'C003'.
  OK: Maya Rosenberg completed class 'C004'.
  OK: Maya Rosenberg completed class 'C005'.
  OK: Maya Rosenberg completed class 'C001'.
  OK: Tom Alon completed class 'C002'.
  OK: Tom Alon completed class 'C005'.
  OK: Shira Dayan completed class 'C001'.


### Step 10: Generate All Reports

In [24]:
# ============================================================
# STEP 10A: Schedule Report
# ============================================================

gym.generate_schedule_report()


  Weekly Schedule Report - FitZone Beer-Sheva

  Sunday:
    08:00-09:00  Morning Yoga | Trainer: Dana Cohen | Enrolled: 3/20
    10:00-11:00  Power Spinning | Trainer: Yossi Levi | Enrolled: 3/12

  Monday:
    09:00-10:00  Pilates Core | Trainer: Dana Cohen | Enrolled: 2/15

  Tuesday:
    17:00-18:00  CrossFit Blast | Trainer: Yossi Levi | Enrolled: 3/3

  Wednesday:
    18:00-19:00  Boxing Fundamentals | Trainer: Noa Shapira | Enrolled: 3/10

  --------------------------------------------------
  No scheduling conflicts detected.


In [25]:
# ============================================================
# STEP 10B: Membership Report
# ============================================================

gym.generate_membership_report()


  Membership Report - FitZone Beer-Sheva
  Total members: 7

  By plan type:
    - monthly: 3
    - yearly: 2
    - premium: 2

  By fitness level:
    - Level 1: 2
    - Level 2: 1
    - Level 3: 2
    - Level 4: 1
    - Level 5: 1

  Active subscriptions: 7
  Inactive subscriptions: 0


In [26]:
# ============================================================
# STEP 10C: Payment Reports - one per member
# ============================================================

for member in gym.members:
    member.generate_payment_report()


  Payment Report: Ido Malach (M001)
  Plan: monthly
  Base price: 250 NIS
  Classes attended: 2
  No-show penalties (1 x 25 NIS): 25 NIS
  -------------------------
  Total due: 275 NIS

  Payment Report: Yonatan Dolman (M002)
  Plan: monthly
  Base price: 250 NIS
  Classes attended: 3
  -------------------------
  Total due: 250 NIS

  Payment Report: Etan Cohen (M003)
  Plan: yearly
  Base price: 200 NIS
  Classes attended: 0
  -------------------------
  Total due: 200 NIS

  Payment Report: Maya Rosenberg (M004)
  Plan: yearly
  Base price: 200 NIS
  Classes attended: 4
  -------------------------
  Total due: 200 NIS

  Payment Report: Tom Alon (M005)
  Plan: premium
  Base price: 350 NIS
  Classes attended: 2
  Classes included: unlimited
  -------------------------
  Total due: 350 NIS

  Payment Report: Shira Dayan (M006)
  Plan: premium
  Base price: 350 NIS
  Classes attended: 1
  Classes included: unlimited
  No-show penalties (1 x 25 NIS): 25 NIS
  ------------------------

In [27]:
# ============================================================
# STEP 10D: Trainer Statistics
# ============================================================

for trainer in gym.trainers:
    trainer.analyze_training_statistics()


  Trainer Statistics: Dana Cohen (T001)
  Experience: 5 years
  Specialties: yoga, pilates
  Classes assigned: 2/5
  Classes by category:
    - yoga: 1
    - pilates: 1
  Total teaching hours/week: 2.0
  Average class size: 2.5
  Personal training sessions: 1 (completed: 0)

  Trainer Statistics: Yossi Levi (T002)
  Experience: 8 years
  Specialties: spinning, crossfit, strength
  Classes assigned: 2/5
  Classes by category:
    - spinning: 1
    - crossfit: 1
  Total teaching hours/week: 2.0
  Average class size: 3.0
  Personal training sessions: 1 (completed: 1)

  Trainer Statistics: Noa Shapira (T003)
  Experience: 3 years
  Specialties: boxing, crossfit
  Classes assigned: 1/5
  Classes by category:
    - boxing: 1
  Total teaching hours/week: 1.0
  Average class size: 3.0
  Personal training sessions: 0 (completed: 0)


In [28]:
# ============================================================
# STEP 10E: Equipment Usage Reports
# ============================================================

for eq in gym.equipment_inventory:
    eq.get_usage_report()


--- Equipment Usage Report: Spinning Bikes (E001) ---
  Type: bike
  Condition: good
  Quantity available: 15
  Assigned to 1 class(es):
    - Power Spinning

--- Equipment Usage Report: Yoga Mats (E002) ---
  Type: mat
  Condition: good
  Quantity available: 25
  Assigned to 2 class(es):
    - Pilates Core
    - Morning Yoga

--- Equipment Usage Report: Free Weights Set (E003) ---
  Type: weights
  Condition: good
  Quantity available: 10
  Assigned to 1 class(es):
    - CrossFit Blast

--- Equipment Usage Report: Punching Bags (E004) ---
  Type: punching_bag
  Condition: good
  Quantity available: 8
  Assigned to 1 class(es):
    - Boxing Fundamentals


In [29]:
# ============================================================
# STEP 10F: Search classes
# ============================================================

print("Search by category:")
gym.search_classes("yoga")

print("\nSearch by trainer name:")
gym.search_classes("yossi")

print("\nSearch by difficulty level:")
gym.search_classes("4")

print("\nSearch with no results:")
gym.search_classes("swimming")

Search by category:

  Search results for 'yoga':
    - FitnessClass(C001: 'Morning Yoga', category=yoga, difficulty=2, trainer=Dana Cohen, schedule=Sunday 08:00, enrolled=3/20)

Search by trainer name:

  Search results for 'yossi':
    - FitnessClass(C002: 'Power Spinning', category=spinning, difficulty=3, trainer=Yossi Levi, schedule=Sunday 10:00, enrolled=3/12)
    - FitnessClass(C004: 'CrossFit Blast', category=crossfit, difficulty=4, trainer=Yossi Levi, schedule=Tuesday 17:00, enrolled=3/3)

Search by difficulty level:

  Search results for '4':
    - FitnessClass(C004: 'CrossFit Blast', category=crossfit, difficulty=4, trainer=Yossi Levi, schedule=Tuesday 17:00, enrolled=3/3)

Search with no results:

  No classes found matching 'swimming'.


[]

### Summary

This simulation demonstrated all the core features of the Gym Management System:

1. **Object creation** — Gym, Trainers, Equipment, FitnessClasses, Members
2. **Class enrollment** — successful enrollment, capacity limits, waitlist management, automatic promotion
3. **Fitness suitability** — members rejected when their fitness level is too low for a class
4. **Schedule conflict detection** — members can't double-book overlapping time slots
5. **Personal training** — booking, conflict prevention, completion with trainer notes
6. **Equipment management** — reservation, malfunction cascade warnings, repair
7. **Reports** — weekly schedule (with conflict flags), membership breakdown, payment per member, trainer statistics, equipment usage, class search

The system is deterministic for now, but the `Equipment` class is designed to support probabilistic breakdowns in a future extension (e.g., using sampling algorithms from weeks 3-4).

---

## AI Usage Declaration

**Tool used:** Claude (Anthropic) — conversational AI assistant

**How we used it:**

We designed the system ourselves — the 7 classes, the relationships between them, the pricing model, the schedule format, and the simulation scenario were all decisions we made as a team. We sketched out the class hierarchy and decided which extra classes to add (Equipment, PersonalTrainingSession, Subscription) based on what made sense for a real gym and what would show meaningful OOP interactions.

Once the design was settled, we used Claude to help with the actual coding — translating our design into working Python. We described what each class should do, what fields and methods it needs, and Claude helped write the implementation. We also used it to write the markdown explanation cells and to make the print messages consistent across all classes (which is why the output formatting is uniform throughout — that's the AI's influence, not something we would have done manually).

Specifically, Claude helped with:
- Writing the class implementations based on our design decisions
- Input validation and error handling patterns (the isinstance checks, ValueError/TypeError raises)
- The overlap detection logic for schedule conflicts (converting times to minutes for comparison)
- Structuring the reports (schedule, membership, payment, trainer statistics)
- Writing clear markdown explanations between code sections
- Making sure every method has a usage example, as required

**What we did ourselves:**
- System design: choosing the 7 classes, defining their relationships, deciding what fields and methods each class needs
- Pricing model: the NIS amounts, included classes per plan, penalty amounts
- Simulation scenario: which members join which classes, the waitlist situation, the fitness mismatch cases, the equipment malfunction demo
- Reviewing every line of code to make sure we understand it and can explain it
- Testing the notebook and verifying the output makes sense
- Design tradeoffs (e.g., modeling equipment at the type level vs. individual units — we chose simplicity and documented why)

**How the output was integrated:**
We didn't copy-paste blindly. We reviewed each class as it was written, asked questions about design choices we didn't understand, and made adjustments where the AI's suggestion didn't match what we wanted. The final notebook is a product of our design decisions implemented with AI assistance.